# Mean-Reversion Trading Strategy Using Z-Score
## Quantitative Finance Project — Final Version

A beginner-friendly Python quantitative finance project using Reliance Industries historical data.

**Strategy:** BUY when Z-score < -2; SELL when Z-score > 0 and a position is held.

This final version includes **brokerage and slippage** in the backtest.

## 1. Objective

Build and backtest a simple mean-reversion strategy, calculate return and risk metrics, and compare it with Buy & Hold.

Starting capital: **₹10,000**

> Educational backtest only; not financial advice.

In [ ]:
# In Google Colab, run this once if required:
# !pip install yfinance

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Download Historical Data

We use Reliance Industries (`RELIANCE.NS`) and daily closing prices.

In [ ]:
data = yf.download(
    "RELIANCE.NS",
    start="2024-01-01",
    end="2026-01-01",
    auto_adjust=True
)

if isinstance(data.columns, pd.MultiIndex):
    data = data.xs("RELIANCE.NS", axis=1, level=1)

data = data[["Close", "High", "Low", "Open", "Volume"]].copy()

print(data.head())
print("Number of trading days:", len(data))

## 3. 20-Day SMA

The 20-day Simple Moving Average is the average closing price of the latest 20 trading days.

In [ ]:
data["SMA20"] = data["Close"].rolling(20).mean()
print(data[["Close", "SMA20"]].head(25))

## 4. 20-Day Standard Deviation

Standard deviation measures how much recent prices vary around their average.

In [ ]:
data["STD20"] = data["Close"].rolling(20).std()
print(data[["Close", "SMA20", "STD20"]].head(25))

## 5. Z-Score

Z-score measures how far the current price is from its 20-day average in standard-deviation units.

**Z = (Price − SMA20) / STD20**

Negative Z-score → price is below its recent average.

Positive Z-score → price is above its recent average.

In [ ]:
data["Z_score"] = (data["Close"] - data["SMA20"]) / data["STD20"]
print(data[["Close", "SMA20", "STD20", "Z_score"]].dropna().head(10))

## 6. Trading Rules

**BUY:** Z-score < -2 and we do not already hold the stock.

**SELL:** Z-score > 0 and we currently hold the stock.

**HOLD:** Otherwise.

The position variable prevents repeated SELL signals when no stock is owned.

In [ ]:
data["Action"] = "HOLD"
data["Position"] = 0
position = 0

for i in range(len(data)):
    z = data["Z_score"].iloc[i]
    action = "HOLD"

    if not pd.isna(z) and z < -2 and position == 0:
        action = "BUY"
        position = 1
    elif not pd.isna(z) and z > 0 and position == 1:
        action = "SELL"
        position = 0

    data.iloc[i, data.columns.get_loc("Action")] = action
    data.iloc[i, data.columns.get_loc("Position")] = position

print(data[["Close", "Z_score", "Action", "Position"]].tail(30))

## 7. Brokerage and Slippage Assumptions

A real trade is not free.

For this educational model:

- **Brokerage = 0.05% per side**
- **Slippage = 0.10% per side**

These are project assumptions, not the exact current charges of a particular broker.

For BUY:
**Execution price = Market price × (1 + slippage)**

For SELL:
**Execution price = Market price × (1 − slippage)**

Brokerage is then charged on the executed trade value.

In [ ]:
STARTING_CAPITAL = 10000.0
BROKERAGE_RATE = 0.0005   # 0.05% per side
SLIPPAGE_RATE = 0.0010    # 0.10% per side

print("Starting capital: ₹", STARTING_CAPITAL)
print("Brokerage per side:", BROKERAGE_RATE * 100, "%")
print("Slippage per side:", SLIPPAGE_RATE * 100, "%")

## 8. Backtest With Brokerage and Slippage

BUY:
1. Increase the execution price for slippage.
2. Calculate affordable shares after brokerage.
3. Deduct the trade cost.

SELL:
1. Reduce the execution price for slippage.
2. Sell the shares.
3. Deduct brokerage.

The portfolio is marked to market every day.

In [ ]:
capital = STARTING_CAPITAL
shares = 0.0

trade_log = []
equity = []
daily_brokerage = []
daily_slippage = []

for date, row in data.iterrows():
    market_price = float(row["Close"])
    action = row["Action"]

    brokerage = 0.0
    slippage_cost = 0.0

    if action == "BUY" and shares == 0:
        execution_price = market_price * (1 + SLIPPAGE_RATE)

        shares = capital / (
            execution_price * (1 + BROKERAGE_RATE)
        )

        gross_value = shares * execution_price
        brokerage = gross_value * BROKERAGE_RATE
        capital = capital - gross_value - brokerage

        slippage_cost = shares * (execution_price - market_price)

        trade_log.append({
            "Date": date,
            "Action": "BUY",
            "Market Price": market_price,
            "Execution Price": execution_price,
            "Shares": shares,
            "Brokerage": brokerage,
            "Slippage Cost": slippage_cost
        })

    elif action == "SELL" and shares > 0:
        execution_price = market_price * (1 - SLIPPAGE_RATE)

        gross_value = shares * execution_price
        brokerage = gross_value * BROKERAGE_RATE
        capital = gross_value - brokerage

        slippage_cost = shares * (market_price - execution_price)

        trade_log.append({
            "Date": date,
            "Action": "SELL",
            "Market Price": market_price,
            "Execution Price": execution_price,
            "Shares": shares,
            "Brokerage": brokerage,
            "Slippage Cost": slippage_cost
        })

        shares = 0.0

    current_equity = capital if shares == 0 else capital + shares * market_price

    equity.append(current_equity)
    daily_brokerage.append(brokerage)
    daily_slippage.append(slippage_cost)

data["Equity"] = equity
data["Brokerage"] = daily_brokerage
data["Slippage_Cost"] = daily_slippage

trades = pd.DataFrame(trade_log)

print(trades.to_string(index=False))

## 9. Strategy Performance

In [ ]:
strategy_final_capital = float(data["Equity"].iloc[-1])
strategy_profit = strategy_final_capital - STARTING_CAPITAL
strategy_return = strategy_profit / STARTING_CAPITAL * 100

total_brokerage = trades["Brokerage"].sum()
total_slippage = trades["Slippage Cost"].sum()

print(f"Final Capital: ₹{strategy_final_capital:,.2f}")
print(f"Total Profit: ₹{strategy_profit:,.2f}")
print(f"Total Return: {strategy_return:.2f}%")
print(f"Total Brokerage: ₹{total_brokerage:,.2f}")
print(f"Estimated Slippage Cost: ₹{total_slippage:,.2f}")

## 10. Buy & Hold Benchmark With Costs

Buy & Hold is also charged for its initial purchase so that the comparison is more consistent.

It buys once and then holds the stock.

In [ ]:
first_market_price = float(data["Close"].iloc[0])

bh_execution_price = first_market_price * (1 + SLIPPAGE_RATE)

bh_shares = STARTING_CAPITAL / (
    bh_execution_price * (1 + BROKERAGE_RATE)
)

bh_gross_cost = bh_shares * bh_execution_price
bh_brokerage = bh_gross_cost * BROKERAGE_RATE
bh_slippage = bh_shares * (bh_execution_price - first_market_price)

data["BuyHold_Equity"] = bh_shares * data["Close"]

bh_final_capital = float(data["BuyHold_Equity"].iloc[-1])
bh_return = (bh_final_capital - STARTING_CAPITAL) / STARTING_CAPITAL * 100

print(f"Buy & Hold Final Capital: ₹{bh_final_capital:,.2f}")
print(f"Buy & Hold Return: {bh_return:.2f}%")
print(f"Initial Buy Brokerage: ₹{bh_brokerage:,.2f}")
print(f"Estimated Initial Slippage: ₹{bh_slippage:,.2f}")

## 11. Maximum Drawdown

Maximum drawdown measures the largest fall from a previous portfolio peak.

**Drawdown = (Current Equity − Previous Peak) / Previous Peak**

In [ ]:
strategy_peak = data["Equity"].cummax()
data["Strategy_Drawdown"] = (data["Equity"] - strategy_peak) / strategy_peak

bh_peak = data["BuyHold_Equity"].cummax()
data["BH_Drawdown"] = (data["BuyHold_Equity"] - bh_peak) / bh_peak

strategy_max_drawdown = data["Strategy_Drawdown"].min()
bh_max_drawdown = data["BH_Drawdown"].min()

print(f"Strategy Maximum Drawdown: {strategy_max_drawdown * 100:.2f}%")
print(f"Buy & Hold Maximum Drawdown: {bh_max_drawdown * 100:.2f}%")

## 12. Volatility and Sharpe Ratio

Daily volatility measures variability in daily portfolio returns.

For this simple project, the risk-free rate is assumed to be zero:

**Sharpe = Mean Daily Return / Std Daily Return × √252**

In [ ]:
strategy_returns = data["Equity"].pct_change().dropna()
bh_returns = data["BuyHold_Equity"].pct_change().dropna()

strategy_volatility = strategy_returns.std()
bh_volatility = bh_returns.std()

strategy_sharpe = strategy_returns.mean() / strategy_returns.std() * np.sqrt(252)
bh_sharpe = bh_returns.mean() / bh_returns.std() * np.sqrt(252)

print(f"Strategy Daily Volatility: {strategy_volatility * 100:.4f}%")
print(f"Buy & Hold Daily Volatility: {bh_volatility * 100:.4f}%")
print(f"Strategy Annualized Sharpe: {strategy_sharpe:.3f}")
print(f"Buy & Hold Annualized Sharpe: {bh_sharpe:.3f}")

## 13. Equity Curve

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(data.index, data["Equity"], label="Mean-Reversion Strategy")
plt.plot(data.index, data["BuyHold_Equity"], label="Buy & Hold")
plt.title("Strategy vs Buy & Hold")
plt.xlabel("Date")
plt.ylabel("Portfolio Value (₹)")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 14. Drawdown Comparison

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(data.index, data["Strategy_Drawdown"] * 100, label="Mean-Reversion Strategy")
plt.plot(data.index, data["BH_Drawdown"] * 100, label="Buy & Hold")
plt.title("Drawdown Comparison")
plt.xlabel("Date")
plt.ylabel("Drawdown (%)")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 15. Z-Score and Trading Thresholds

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(data.index, data["Z_score"], label="Z-score")
plt.axhline(-2, linestyle="--", label="BUY threshold (-2)")
plt.axhline(0, linestyle="--", label="SELL threshold (0)")
plt.title("Z-Score and Trading Thresholds")
plt.xlabel("Date")
plt.ylabel("Z-score")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 16. Final Performance Comparison

In [ ]:
comparison = pd.DataFrame({
    "Metric": [
        "Starting Capital",
        "Final Capital",
        "Total Return (%)",
        "Maximum Drawdown (%)",
        "Daily Volatility (%)",
        "Annualized Sharpe Ratio"
    ],
    "Our Strategy": [
        STARTING_CAPITAL,
        strategy_final_capital,
        strategy_return,
        strategy_max_drawdown * 100,
        strategy_volatility * 100,
        strategy_sharpe
    ],
    "Buy & Hold": [
        STARTING_CAPITAL,
        bh_final_capital,
        bh_return,
        bh_max_drawdown * 100,
        bh_volatility * 100,
        bh_sharpe
    ]
})

print("================ PERFORMANCE COMPARISON ================\n")
print(comparison.to_string(index=False))

## 17. Cost Summary

In [ ]:
cost_summary = pd.DataFrame({
    "Cost": [
        "Strategy Brokerage",
        "Strategy Slippage",
        "Buy & Hold Initial Brokerage",
        "Buy & Hold Initial Slippage"
    ],
    "Amount (₹)": [
        total_brokerage,
        total_slippage,
        bh_brokerage,
        bh_slippage
    ]
})

print(cost_summary.to_string(index=False))

## 18. Conclusion

The project converts a statistical idea into a systematic trading strategy:

**Historical Data → SMA → Standard Deviation → Z-score → Trading Signals → Position Tracking → Costs → Backtest → Risk Analysis → Benchmark**

The important lesson is that return alone is not enough. Brokerage and slippage reduce theoretical profits, while drawdown, volatility and Sharpe Ratio help evaluate risk.

### Limitations
1. Brokerage and slippage are simplified project assumptions.
2. Taxes and other market charges are not modeled individually.
3. Only one stock and one historical period are tested.
4. Historical performance does not guarantee future performance.
5. The parameters should be tested on separate out-of-sample data.
6. This is not a live trading system.

### Future Improvements
- Test multiple stocks.
- Add exact broker-specific charges.
- Add out-of-sample testing.
- Test different SMA windows.
- Test different entry/exit thresholds.
- Add risk-management rules.